In [1]:
import os
import copy
import random
import collections
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.sparse.csgraph import connected_components
import matplotlib.pyplot as plt

# Scikit-learn Preprocessing, Splitting, and Metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, balanced_accuracy_score, matthews_corrcoef, precision_recall_curve, auc, brier_score_loss
from sklearn.calibration import calibration_curve, CalibrationDisplay

# Imbalanced Learning Frameworks
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek

# Machine Learning & Deep Learning Frameworks
from xgboost import XGBClassifier
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Explainable AI (XAI) Libraries
import shap
import lime
import lime.lime_tabular

import preprocess

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    # Ensure fully deterministic behavior in PyTorch backends
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [3]:
filepath = "data/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv"
df = preprocess.load_and_clean_dataset(filepath)

#change to a binary label
df['Label'] = df['Label'].astype(str).str.strip().str.upper()
df['Label'] = df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

X = df.drop(columns=['Label'])
y = df['Label'].values

print(f"[*] Cleaned Feature matrix shape: {X.shape}")
print(f"[*] Target distribution: Benign (0) = {np.sum(y == 0)}, Attack (1) = {np.sum(y == 1)}")


[*] Loading raw dataset from: data/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
[*] Dataset Ingestion & Cleaning Audit:
    -> Raw Rows Ingested:          458,968
    -> Empty CSV Padding Purged:   288,602
    -> Valid Network Flows:        170,366
    -> Invalid Flows Dropped (Inf/NaN): 135
    -> Final Usable Flows:         170,231
[*] Cleaned Feature matrix shape: (170231, 78)
[*] Target distribution: Benign (0) = 168051, Attack (1) = 2180


In [4]:
# Step A: Split off 30% of the data into a temporary block, stratifying to preserve class ratios
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# Step B: Split the temporary block evenly to yield a 15% Validation set and a 15% Test set
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("==================== DATA SPLIT SUMMARY ====================")
print(f"Training Set (70%):   X = {X_train.shape}, y = {y_train.shape}")
print(f"Validation Set (15%): X = {X_val.shape}, y = {y_val.shape}")
print(f"Test Set (15%):       X = {X_test.shape}, y = {y_test.shape}")

==================== DATA SPLIT SUMMARY ====================
Training Set (70%):   X = (119161, 78), y = (119161,)
Validation Set (15%): X = (25535, 78), y = (25535,)
Test Set (15%):       X = (25535, 78), y = (25535,)


In [5]:
# 1. Fit scaler ONLY on training data
scaler = MinMaxScaler()
scaler.fit(X_train)

X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

# 2. Correlated Feature Groups (Supervisor point 15 & 16)
corr_matrix = X_train_scaled.corr(method="spearman")
# Get upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
# Find columns with correlation > 0.90
to_drop = [column for column in upper.columns if any(upper[column].abs() > 0.90)]

print(f"[*] Features identified for potential removal (>0.90 correlation): {len(to_drop)}")

[*] Features identified for potential removal (>0.90 correlation): 39


In [6]:
# Initialize SMOTE-Tomek with a fixed seed for strict reproducibility
smote_tomek = SMOTETomek(random_state=42)

# Resample ONLY the training set
X_train_resampled, y_train_resampled = smote_tomek.fit_resample(
    X_train_scaled,
    y_train
)

print("================= RESAMPLING SUMMARY =================")
print(f"Original Training Class Ratios: 0 = {np.sum(y_train == 0)}, 1 = {np.sum(y_train == 1)}")
print(f"Resampled Training Data Shape:  X = {X_train_resampled.shape}, y = {y_train_resampled.shape}")
print(f"Resampled Training Class Ratios: 0 = {np.sum(y_train_resampled == 0)}, 1 = {np.sum(y_train_resampled == 1)}")

================= RESAMPLING SUMMARY =================
Original Training Class Ratios: 0 = 117635, 1 = 1526
Resampled Training Data Shape:  X = (235254, 78), y = (235254,)
Resampled Training Class Ratios: 0 = 117627, 1 = 117627


In [7]:
print("[*] Initializing XGBoost Classifier...")

# Initialize XGBoost with strict reproducibility parameters
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=0.1,
    random_state=42,
    use_label_encoder=False,
    early_stopping_rounds=15
)

# Fit model with early stopping monitored against the validation split
xgb_model.fit(
    X_train_resampled, 
    y_train_resampled,
    eval_set=[(X_val_scaled, y_val)],
    verbose=False
)

print(f"[*] XGBoost training complete.")
print(f"    -> Best Iteration: {xgb_model.best_iteration}")

[*] Initializing XGBoost Classifier...


c:\Users\Moritz\Documents\Uni\bachlor_thesis\ids-xai-thesis\thesis_env\Lib\site-packages\xgboost\callback.py:385: UserWarning: [16:53:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[*] XGBoost training complete.
    -> Best Iteration: 398


In [8]:
class RobustNetworkSecurityDNN(nn.Module):
    def __init__(self, input_dim):
        super(RobustNetworkSecurityDNN, self).__init__()
        
        # Layer 1: Input to Hidden 1
        self.fc1 = nn.Linear(input_dim, 128)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(p=0.30)
        
        # Layer 2: Hidden 1 to Hidden 2
        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(p=0.30)
        
        # Layer 3: Hidden 2 to Output Layer
        self.fc3 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        x = self.dropout1(self.relu1(self.fc1(x)))
        x = self.dropout2(self.relu2(self.fc2(x)))
        x = self.sigmoid(self.fc3(x))
        return x

# Set calculation device backend
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_features_count = X_train_resampled.shape[1]

# Instantiate model architecture
model_dnn = RobustNetworkSecurityDNN(input_dim=input_features_count).to(device)
print(f"[*] PyTorch Network mapped successfully onto target hardware device: {device.type.upper()}")

[*] PyTorch Network mapped successfully onto target hardware device: CPU


In [9]:
# Convert DataFrames/Arrays to PyTorch multi-dimensional tensors
train_dataset = TensorDataset(
    torch.FloatTensor(X_train_resampled.values),
    torch.FloatTensor(y_train_resampled).unsqueeze(1)
)
val_x_tensor = torch.FloatTensor(X_val_scaled.values).to(device)
val_y_tensor = torch.FloatTensor(y_val).unsqueeze(1).to(device)

# Configure data loader iterations
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)

# Set optimizer equations and standard binary loss scoring
criterion = nn.BCELoss()
optimizer = optim.Adam(model_dnn.parameters(), lr=0.001)

# Early Stopping parameters
patience = 10
best_val_loss = float('inf')
best_model_weights = None
patience_counter = 0
max_epochs = 150

print("[*] Initiating DNN optimization loop...")
for epoch in range(1, max_epochs + 1):
    model_dnn.train()
    running_loss = 0.0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model_dnn(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_x.size(0)
        
    # Validation Evaluation Phase (Zero Leakage Check)
    model_dnn.eval()
    with torch.no_grad():
        val_outputs = model_dnn(val_x_tensor)
        val_loss = criterion(val_outputs, val_y_tensor).item()
        
    epoch_train_loss = running_loss / len(train_dataset)
    
    # Early stopping criteria tracking
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_weights = copy.deepcopy(model_dnn.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        
    if patience_counter >= patience:
        print(f"[*] Early stopping triggered at Epoch {epoch}. Overfitting threshold neutralized.")
        break

# Roll back parameter matrix states to the optimal captured validation validation loss weights
if best_model_weights is not None:
    model_dnn.load_state_dict(best_model_weights)
print(f"[*] Restored optimal model configurations. Best Validation Loss: {best_val_loss:.5f}")

C:\Users\Moritz\AppData\Local\Temp\ipykernel_26936\4040257355.py:6: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  val_x_tensor = torch.FloatTensor(X_val_scaled.values).to(device)


[*] Initiating DNN optimization loop...
[*] Early stopping triggered at Epoch 21. Overfitting threshold neutralized.
[*] Restored optimal model configurations. Best Validation Loss: 0.04361


In [10]:
# Helper function to generate clean inference probability matrices from our DNN
def get_dnn_probabilities(df_input):
    model_dnn.eval()
    with torch.no_grad():
        tensor_input = torch.FloatTensor(df_input.values).to(device)
        probs = model_dnn(tensor_input).cpu().numpy().flatten()
    return probs

# Step A: Collect raw feature inference array probabilities from validation subsets
xgb_val_probs = xgb_model.predict_proba(X_val_scaled)[:, 1]
dnn_val_probs = get_dnn_probabilities(X_val_scaled)

# Step B: Declare target tuning thresholds range
thresholds_pool = np.arange(0.50, 0.96, 0.05)

best_xgb_threshold = 0.50
best_xgb_f1 = 0.0
best_dnn_threshold = 0.50
best_dnn_f1 = 0.0

print("================= VAL THRESHOLD CALIBRATION =================")
for t in thresholds_pool:
    # Evaluate XGBoost arrays
    xgb_preds = (xgb_val_probs >= t).astype(int)
    xgb_f1 = f1_score(y_val, xgb_preds, zero_division=0)
    if xgb_f1 > best_xgb_f1:
        best_xgb_f1 = xgb_f1
        best_xgb_threshold = t
        
    # Evaluate DNN arrays
    dnn_preds = (dnn_val_probs >= t).astype(int)
    dnn_f1 = f1_score(y_val, dnn_preds, zero_division=0)
    if dnn_f1 > best_dnn_f1:
        best_dnn_f1 = dnn_f1
        best_dnn_threshold = t
        
    print(f"Threshold: {t:.2f} | XGB F1: {xgb_f1:.4f} | DNN F1: {dnn_f1:.4f}")

print("\n[*] Calibrated Decision Parameter Options Frozen:")
print(f"    -> Selected Frozen XGBoost Threshold: {best_xgb_threshold:.2f} (Val F1: {best_xgb_f1:.4f})")
print(f"    -> Selected Frozen DNN Threshold:     {best_dnn_threshold:.2f} (Val F1: {best_dnn_f1:.4f})")

================= VAL THRESHOLD CALIBRATION =================
Threshold: 0.50 | XGB F1: 0.9954 | DNN F1: 0.5195
Threshold: 0.55 | XGB F1: 0.9954 | DNN F1: 0.5304
Threshold: 0.60 | XGB F1: 0.9954 | DNN F1: 0.5450
Threshold: 0.65 | XGB F1: 0.9954 | DNN F1: 0.6054
Threshold: 0.70 | XGB F1: 0.9954 | DNN F1: 0.6681
Threshold: 0.75 | XGB F1: 0.9970 | DNN F1: 0.7687
Threshold: 0.80 | XGB F1: 0.9970 | DNN F1: 0.8626
Threshold: 0.85 | XGB F1: 0.9969 | DNN F1: 0.8555
Threshold: 0.90 | XGB F1: 0.9969 | DNN F1: 0.8547
Threshold: 0.95 | XGB F1: 0.9985 | DNN F1: 0.8571

[*] Calibrated Decision Parameter Options Frozen:
    -> Selected Frozen XGBoost Threshold: 0.95 (Val F1: 0.9985)
    -> Selected Frozen DNN Threshold:     0.80 (Val F1: 0.8626)


In [11]:
# Step A: Extract test predictions using frozen configurations
xgb_test_probs = xgb_model.predict_proba(X_test_scaled)[:, 1]
dnn_test_probs = get_dnn_probabilities(X_test_scaled)

xgb_test_preds = (xgb_test_probs >= best_xgb_threshold).astype(int)
dnn_test_preds = (dnn_test_probs >= best_dnn_threshold).astype(int)

# Step B: Print formal validation summaries for your thesis report
print("=================== FINAL FROZEN XGBOOST REPORT ===================")
print(f"Applied Decision Threshold: {best_xgb_threshold:.2f}")
print(confusion_matrix(y_test, xgb_test_preds))
print(classification_report(y_test, xgb_test_preds, digits=4))

print("\n==================== FINAL FROZEN DNN REPORT ====================")
print(f"Applied Decision Threshold: {best_dnn_threshold:.2f}")
print(confusion_matrix(y_test, dnn_test_preds))
print(classification_report(y_test, dnn_test_preds, digits=4))

=================== FINAL FROZEN XGBOOST REPORT ===================
Applied Decision Threshold: 0.95
[[25207     1]
 [    4   323]]
              precision    recall  f1-score   support

           0     0.9998    1.0000    0.9999     25208
           1     0.9969    0.9878    0.9923       327

    accuracy                         0.9998     25535
   macro avg     0.9984    0.9939    0.9961     25535
weighted avg     0.9998    0.9998    0.9998     25535


==================== FINAL FROZEN DNN REPORT ====================
Applied Decision Threshold: 0.80
[[25126    82]
 [   19   308]]
              precision    recall  f1-score   support

           0     0.9992    0.9967    0.9980     25208
           1     0.7897    0.9419    0.8591       327

    accuracy                         0.9960     25535
   macro avg     0.8945    0.9693    0.9286     25535
weighted avg     0.9966    0.9960    0.9962     25535



In [12]:
# Standardized probability wrappers for the explainers
def xgb_predict_proba_wrapper(x_numpy):
    # Map numpy matrix rows directly back to pandas format to preserve column naming indices natively
    df_temp = pd.DataFrame(x_numpy, columns=X_train_scaled.columns)
    return xgb_model.predict_proba(df_temp)

def dnn_predict_proba_wrapper(x_numpy):
    df_temp = pd.DataFrame(x_numpy, columns=X_train_scaled.columns)
    probs_class_1 = get_dnn_probabilities(df_temp)
    probs_class_0 = 1.0 - probs_class_1
    return np.column_stack((probs_class_0, probs_class_1))

# step 6: correlation analysis

## phase 1 correlation grouping

In [13]:

def compute_correlation_groups(df_train, threshold=0.90):
    """
    Computes Spearman correlation matrix and groups features into clusters
    using connected components analysis based on a correlation threshold.
    Selects group representatives based on the highest average correlation 
    within the group.
    """
    print(f"[*] Calculating pairwise Spearman correlation matrix for {df_train.shape[1]} features...")
    corr_matrix = df_train.corr(method='spearman').abs()
    
    # Create an adjacency matrix where 1 indicates correlation >= threshold
    adj_matrix = (corr_matrix >= threshold).astype(int)
    
    # Find connected components in the correlation graph
    n_components, labels = connected_components(adj_matrix.values, directed=False)
    
    feature_names = df_train.columns
    group_to_features = collections.defaultdict(list)
    for idx, label in enumerate(labels):
        group_to_features[label].append(feature_names[idx])
        
    feature_to_group = {}
    final_groups = {}
    representative_features = {}
    
    group_counter = 0
    independent_counter = 0
    
    sorted_groups = sorted(group_to_features.values(), key=len, reverse=True)
    
    for features in sorted_groups:
        if len(features) > 1:
            group_name = f"Correlation_Group_{group_counter}"
            group_counter += 1
            
            # Justified Representative Selection: Feature with highest mean correlation to group members
            sub_corr = corr_matrix.loc[features, features]
            rep_feat = sub_corr.mean(axis=1).idxmax()
        else:
            group_name = f"Independent_{independent_counter}"
            independent_counter += 1
            rep_feat = features[0]
            
        final_groups[group_name] = sorted(features)
        representative_features[group_name] = rep_feat
        
        for feat in features:
            feature_to_group[feat] = group_name
            
    print(f"[+] Discovered {group_counter} highly correlated feature groups and {independent_counter} independent features.")
    return final_groups, feature_to_group, representative_features

# Run correlation grouping pipeline on training set
CORRELATION_THRESHOLD = 0.90
group_to_features, feature_to_group, group_representatives = compute_correlation_groups(
    X_train_scaled, 
    threshold=CORRELATION_THRESHOLD
)

print("\n" + "="*80)
print(f"   SUMMARY OF DISCOVERED HIGHLY CORRELATED FEATURE GROUPS (Threshold >= {CORRELATION_THRESHOLD})")
print("="*80)

total_grouped_features = 0
for g_name, features in group_to_features.items():
    if "Correlation_Group_" in g_name:
        total_grouped_features += len(features)
        print(f"-> {g_name} ({len(features)} features) | Representative (Highest Intra-Group Corr): '{group_representatives[g_name]}'")
        for feat in features:
            print(f"   - {feat}")
        print("-"*80)

print(f"[*] Analysis complete. Total features absorbed into correlation groups: {total_grouped_features}")
print(f"[*] Remaining standalone independent features: {len(X_train_scaled.columns) - total_grouped_features}")
print("="*80)

[*] Calculating pairwise Spearman correlation matrix for 78 features...
[+] Discovered 13 highly correlated feature groups and 26 independent features.

   SUMMARY OF DISCOVERED HIGHLY CORRELATED FEATURE GROUPS (Threshold >= 0.9)
-> Correlation_Group_0 (10 features) | Representative (Highest Intra-Group Corr): 'Bwd Packet Length Max'
   - Average Packet Size
   - Avg Bwd Segment Size
   - Bwd Packet Length Max
   - Bwd Packet Length Mean
   - Max Packet Length
   - Packet Length Mean
   - Packet Length Std
   - Packet Length Variance
   - Subflow Bwd Bytes
   - Total Length of Bwd Packets
--------------------------------------------------------------------------------
-> Correlation_Group_1 (8 features) | Representative (Highest Intra-Group Corr): 'Total Fwd Packets'
   - Fwd Header Length
   - Fwd Header Length.1
   - Fwd IAT Max
   - Fwd IAT Mean
   - Fwd IAT Total
   - Subflow Fwd Packets
   - Total Fwd Packets
   - act_data_pkt_fwd
--------------------------------------------------

In [14]:
#Helper Functions
feature_list = X_train_scaled.columns.tolist()
num_features_original = len(feature_list)

if 'rng' not in globals():
    rng = np.random.default_rng(seed=42)

def extract_aligned_lime_attributions(instance, predict_fn, num_features, explainer=None):
    active_exp = lime_explainer if explainer is None else explainer
    exp = active_exp.explain_instance(
        data_row=instance,
        predict_fn=predict_fn,
        labels=(1,),
        num_features=num_features
    )
    lime_vector = np.zeros(num_features)
    for feat_idx, weight in exp.local_exp[1]:
        lime_vector[feat_idx] = weight
    return lime_vector, exp.score

def extract_aligned_shap_attributions(instance, explainer):
    shap_vals = explainer.shap_values(instance.reshape(1, -1), nsamples=100)
    if isinstance(shap_vals, list):
        shap_vector = shap_vals[1].flatten()
    elif isinstance(shap_vals, np.ndarray):
        if len(shap_vals.shape) == 3:
            shap_vector = shap_vals[0, :, 1]
        elif len(shap_vals.shape) == 2:
            shap_vector = shap_vals.flatten()
        else:
            shap_vector = shap_vals.flatten()
    else:
        shap_vector = np.array(shap_vals).flatten()
    return shap_vector

def compute_bootstrapped_ci(data_series, n_bootstrap=5000, ci=0.95):
    clean_data = data_series.dropna().values
    if len(clean_data) == 0:
        return 0.0, 0.0, 0.0
    boot_means = []
    for _ in range(n_bootstrap):
        sample = rng.choice(clean_data, size=len(clean_data), replace=True)
        boot_means.append(np.mean(sample))
    
    mean_val = np.mean(clean_data)
    lower_bound = np.percentile(boot_means, ((1.0 - ci) / 2.0) * 100)
    upper_bound = np.percentile(boot_means, (ci + (1.0 - ci) / 2.0) * 100)
    return mean_val, lower_bound, upper_bound

In [15]:
# Helper to sample without replacement safely
def sample_indices(indices, max_n=50):
    if len(indices) == 0:
        return []
    return rng.choice(indices, min(max_n, len(indices)), replace=False).tolist()

# 1. XGBoost Cohorts
xgb_probs = xgb_model.predict_proba(X_test_scaled)[:, 1]
xgb_preds = (xgb_probs >= best_xgb_threshold).astype(int)

xgb_tp = np.where((y_test == 1) & (xgb_preds == 1))[0]
xgb_tn = np.where((y_test == 0) & (xgb_preds == 0))[0]
xgb_fp = np.where((y_test == 0) & (xgb_preds == 1))[0]
xgb_fn = np.where((y_test == 1) & (xgb_preds == 0))[0]

# XGBoost borderline cases: closest to XGBoost's own threshold
xgb_margin = np.abs(xgb_probs - best_xgb_threshold)
xgb_borderline = np.argsort(xgb_margin)[:100]

model_cohorts = {
    'XGBoost': {
        'True Positives': sample_indices(xgb_tp),
        'True Negatives': sample_indices(xgb_tn),
        'False Positives': sample_indices(xgb_fp),
        'False Negatives': sample_indices(xgb_fn),
        'Borderline Cases': sample_indices(xgb_borderline)
    }
}

# 2. DNN Cohorts
dnn_probs = get_dnn_probabilities(X_test_scaled)
dnn_preds = (dnn_probs >= best_dnn_threshold).astype(int)

dnn_tp = np.where((y_test == 1) & (dnn_preds == 1))[0]
dnn_tn = np.where((y_test == 0) & (dnn_preds == 0))[0]
dnn_fp = np.where((y_test == 0) & (dnn_preds == 1))[0]
dnn_fn = np.where((y_test == 1) & (dnn_preds == 0))[0]

# DNN borderline cases: closest to DNN's own threshold
dnn_margin = np.abs(dnn_probs - best_dnn_threshold)
dnn_borderline = np.argsort(dnn_margin)[:100]

model_cohorts['DNN'] = {
    'True Positives': sample_indices(dnn_tp),
    'True Negatives': sample_indices(dnn_tn),
    'False Positives': sample_indices(dnn_fp),
    'False Negatives': sample_indices(dnn_fn),
    'Borderline Cases': sample_indices(dnn_borderline)
}

print("=================== MODEL-SPECIFIC COHORT SAMPLE COUNTS (N) ===================")
for model_name, cohorts in model_cohorts.items():
    print(f"[{model_name}]")
    for c_name, c_indices in cohorts.items():
        print(f"  -> {c_name}: N = {len(c_indices)}")
print("===============================================================================")

=================== MODEL-SPECIFIC COHORT SAMPLE COUNTS (N) ===================
[XGBoost]
  -> True Positives: N = 50
  -> True Negatives: N = 50
  -> False Positives: N = 1
  -> False Negatives: N = 4
  -> Borderline Cases: N = 50
[DNN]
  -> True Positives: N = 50
  -> True Negatives: N = 50
  -> False Positives: N = 50
  -> False Negatives: N = 19
  -> Borderline Cases: N = 50


In [16]:
#Baseline Explainers with Primary Continuous Setup
# Initialize LIME Explainer with primary continuous setup (KW=2.5)
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_scaled.values,
    feature_names=feature_list,
    class_names=["Benign", "Web Attack"],
    mode="classification",
    discretize_continuous=False,
    kernel_width=2.5,
    random_state=42
)

# Initialize SHAP Kernel Explainers
shap_background_baseline = shap.sample(X_train_scaled, 100, random_state=42)
shap_explainer_xgb = shap.KernelExplainer(model=xgb_predict_proba_wrapper, data=shap_background_baseline)
shap_explainer_dnn = shap.KernelExplainer(model=dnn_predict_proba_wrapper, data=shap_background_baseline)

# Build correlation group dimension mappings and size normalization arrays
unique_groups_list = sorted(list(set(feature_to_group.values())))
num_groups_total = len(unique_groups_list)
group_name_to_index = {g_name: idx for idx, g_name in enumerate(unique_groups_list)}
group_sizes = np.array([len(group_to_features[g_name]) for g_name in unique_groups_list])

print(f"[+] Baseline Explainers initialized: Continuous LIME (KW=2.5) over {num_groups_total} correlation group dimensions.")

[+] Baseline Explainers initialized: Continuous LIME (KW=2.5) over 39 correlation group dimensions.


In [17]:
#Group Attribution Extraction with Multi-K Overlap Suite
grouped_agreement_records = []
print("[*] Extracting attributions and computing MEAN attributions for correlation groups...")

model_configs = [
    ('XGBoost', xgb_predict_proba_wrapper, shap_explainer_xgb),
    ('DNN', dnn_predict_proba_wrapper, shap_explainer_dnn)
]

for model_label, predict_fn, shap_exp in model_configs:
    print(f"\n[*] Evaluating cohorts for model: {model_label}")
    for cohort_name, indices in model_cohorts[model_label].items():
        if not indices:
            continue
        sample_count = len(indices)
        print(f"    -> Auditing cohort: {cohort_name} (N = {sample_count})...")
        
        for test_idx in indices:
            instance_vector = X_test_scaled.iloc[test_idx].values
            
            lime_v, lime_r2 = extract_aligned_lime_attributions(instance_vector, predict_fn, num_features_original)
            shap_v = extract_aligned_shap_attributions(instance_vector, shap_exp)
            
            lime_grouped_sum = np.zeros(num_groups_total)
            shap_grouped_sum = np.zeros(num_groups_total)
            
            for f_idx, f_name in enumerate(feature_list):
                g_name = feature_to_group[f_name]
                g_idx = group_name_to_index[g_name]
                lime_grouped_sum[g_idx] += np.abs(lime_v[f_idx])
                shap_grouped_sum[g_idx] += np.abs(shap_v[f_idx])
                
            lime_grouped_mean = lime_grouped_sum / group_sizes
            shap_grouped_mean = shap_grouped_sum / group_sizes
            
            row_metrics = {
                'Cohort': cohort_name,
                'N_Samples': sample_count,
                'Model': model_label,
                'Test_Index': test_idx,
                'LIME_R2': lime_r2
            }
            
            for k in [3, 5, 10]:
                top_lime_g = set(np.argsort(lime_grouped_mean)[-k:])
                top_shap_g = set(np.argsort(shap_grouped_mean)[-k:])
                
                overlap = len(top_lime_g.intersection(top_shap_g))
                union_set = top_lime_g.union(top_shap_g)
                jaccard = overlap / len(union_set) if len(union_set) > 0 else 0.0
                
                row_metrics[f'Mean_Overlap_{k}'] = overlap
                row_metrics[f'Mean_Jaccard_{k}'] = jaccard
                
                # Localized Rank Correlation over the Union of Top-K groups
                union_indices = list(union_set)
                slice_l = lime_grouped_mean[union_indices]
                slice_s = shap_grouped_mean[union_indices]
                
                if np.std(slice_l) == 0 or np.std(slice_s) == 0 or len(union_indices) < 2:
                    row_metrics[f'Union_Spearman_{k}'] = 0.0
                else:
                    rho, _ = stats.spearmanr(slice_l, slice_s)
                    row_metrics[f'Union_Spearman_{k}'] = 0.0 if np.isnan(rho) else rho
                
            grouped_agreement_records.append(row_metrics)

df_grouped_agreement = pd.DataFrame(grouped_agreement_records)
print("[+] Group attribution extraction complete.")

[*] Extracting attributions and computing MEAN attributions for correlation groups...

[*] Evaluating cohorts for model: XGBoost
    -> Auditing cohort: True Positives (N = 50)...


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

[+] Group attribution extraction complete.


In [18]:
#Group Report with 95% Bootstrap CIs
grouped_report_rows = []

for (cohort_name, model_type), group_df in df_grouped_agreement.groupby(['Cohort', 'Model']):
    row_summary = {
        'Cohort': cohort_name, 
        'N_Samples': group_df['N_Samples'].iloc[0],
        'Model': model_type
    }
    for metric in ['LIME_R2', 'Mean_Overlap_3', 'Mean_Overlap_5', 'Mean_Overlap_10', 'Mean_Jaccard_5', 'Mean_Jaccard_10', 'Union_Spearman_10']:
        mean_v, low_v, high_v = compute_bootstrapped_ci(group_df[metric], n_bootstrap=5000)
        row_summary[metric] = f"{mean_v:.3f} [{low_v:.3f}, {high_v:.3f}]"
        
    grouped_report_rows.append(row_summary)

df_grouped_master_report = pd.DataFrame(grouped_report_rows)

print("\n" + "="*160)
print("                           FINAL CORRELATION GROUP-LEVEL AGREEMENT REPORT WITH 95% CIs")
print("="*160)
display(df_grouped_master_report)
print("="*160)


                           FINAL CORRELATION GROUP-LEVEL AGREEMENT REPORT WITH 95% CIs


,Cohort,N_Samples,Model,LIME_R2,Mean_Overlap_3,Mean_Overlap_5,Mean_Overlap_10,Mean_Jaccard_5,Mean_Jaccard_10,Union_Spearman_10
0,Borderline Cases,50,DNN,"0.762 [0.750, 0.775]","1.360 [1.160, 1.560]","2.320 [2.080, 2.580]","4.120 [3.760, 4.460]","0.321 [0.277, 0.366]","0.267 [0.240, 0.294]","-0.005 [-0.075, 0.065]"
1,Borderline Cases,50,XGBoost,"0.769 [0.717, 0.808]","0.840 [0.620, 1.080]","1.660 [1.380, 1.940]","3.220 [2.940, 3.500]","0.218 [0.178, 0.260]","0.196 [0.176, 0.218]","-0.232 [-0.292, -0.171]"
2,False Negatives,19,DNN,"0.682 [0.601, 0.745]","1.053 [0.737, 1.368]","1.842 [1.526, 2.105]","3.737 [3.263, 4.158]","0.234 [0.189, 0.275]","0.235 [0.199, 0.270]","-0.128 [-0.234, -0.024]"
3,False Negatives,4,XGBoost,"0.627 [0.266, 0.897]","0.500 [0.000, 1.000]","1.000 [0.250, 1.750]","3.750 [3.250, 4.000]","0.118 [0.028, 0.215]","0.232 [0.195, 0.250]","-0.319 [-0.548, -0.084]"
4,False Positives,50,DNN,"0.834 [0.824, 0.844]","1.360 [1.140, 1.580]","2.480 [2.220, 2.740]","4.480 [4.120, 4.840]","0.350 [0.304, 0.399]","0.298 [0.268, 0.330]","0.018 [-0.050, 0.080]"
5,False Positives,1,XGBoost,"0.781 [0.781, 0.781]","0.000 [0.000, 0.000]","1.000 [1.000, 1.000]","2.000 [2.000, 2.000]","0.111 [0.111, 0.111]","0.111 [0.111, 0.111]","-0.524 [-0.524, -0.524]"
6,True Negatives,50,DNN,"0.059 [0.052, 0.067]","0.220 [0.100, 0.340]","0.880 [0.660, 1.100]","3.100 [2.760, 3.440]","0.104 [0.078, 0.132]","0.190 [0.165, 0.213]","-0.508 [-0.561, -0.455]"
7,True Negatives,50,XGBoost,"0.037 [0.030, 0.045]","0.860 [0.640, 1.080]","1.700 [1.420, 1.980]","3.920 [3.580, 4.260]","0.223 [0.181, 0.267]","0.252 [0.225, 0.281]","-0.206 [-0.279, -0.131]"
8,True Positives,50,DNN,"0.820 [0.805, 0.832]","1.200 [1.000, 1.420]","2.340 [2.100, 2.580]","4.300 [4.000, 4.620]","0.324 [0.280, 0.368]","0.281 [0.255, 0.308]","-0.011 [-0.079, 0.058]"
9,True Positives,50,XGBoost,"0.804 [0.754, 0.842]","0.860 [0.680, 1.040]","2.120 [1.920, 2.300]","3.760 [3.480, 4.080]","0.279 [0.248, 0.310]","0.237 [0.214, 0.264]","-0.176 [-0.227, -0.126]"


In [19]:


# Step 1: Isolate justified representative feature list
reduced_features_list = []
for g_name, features in group_to_features.items():
    if "Correlation_Group_" in g_name:
        reduced_features_list.append(group_representatives[g_name])
    else:
        reduced_features_list.extend(features)

reduced_features_list = sorted(list(set(reduced_features_list)))
num_features_reduced = len(reduced_features_list)

print(f"[+] Pruned feature space from {len(X_train_scaled.columns)} down to {num_features_reduced} features.")

# Step 2: Extract scaled feature subsets BEFORE resampling
X_train_reduced = X_train_scaled[reduced_features_list].copy()
X_val_reduced = X_val_scaled[reduced_features_list].copy()
X_test_reduced = X_test_scaled[reduced_features_list].copy()

# Step 3: Execute SMOTE-Tomek fresh inside the reduced feature space
print("[*] Re-running SMOTE-Tomek resampling strictly within the reduced feature space...")
smote_tomek_reduced = SMOTETomek(random_state=42)

X_train_resampled_reduced, y_train_resampled_reduced = smote_tomek_reduced.fit_resample(
    X_train_reduced,
    y_train
)

print(f"[+] Resampled Reduced Training Set Shape: X = {X_train_resampled_reduced.shape}, y = {y_train_resampled_reduced.shape}")

[+] Pruned feature space from 78 down to 39 features.
[*] Re-running SMOTE-Tomek resampling strictly within the reduced feature space...
[+] Resampled Reduced Training Set Shape: X = (235104, 39), y = (235104,)


In [20]:


print("[*] Re-training XGBoost on reduced feature space...")

xgb_hyperparams = xgb_model.get_params()
xgb_model_reduced = XGBClassifier(**xgb_hyperparams)

xgb_model_reduced.fit(
    X_train_resampled_reduced,
    y_train_resampled_reduced,
    eval_set=[(X_val_reduced, y_val)],
    verbose=False
)

xgb_val_probs_r = xgb_model_reduced.predict_proba(X_val_reduced)[:, 1]

# Match baseline threshold search procedure across validation probabilities
thresholds_pool = np.arange(0.50, 0.96, 0.05)
best_thresh_xgb_reduced = 0.50
best_f1_xgb_r = 0.0

for t in thresholds_pool:
    preds = (xgb_val_probs_r >= t).astype(int)
    f1 = f1_score(y_val, preds, zero_division=0)
    if f1 > best_f1_xgb_r:
        best_f1_xgb_r = f1
        best_thresh_xgb_reduced = t

print(f"[+] Reduced XGBoost Calibration Complete. Optimal Threshold: {best_thresh_xgb_reduced:.2f} (Val F1: {best_f1_xgb_r:.4f})")

[*] Re-training XGBoost on reduced feature space...


c:\Users\Moritz\Documents\Uni\bachlor_thesis\ids-xai-thesis\thesis_env\Lib\site-packages\xgboost\callback.py:385: UserWarning: [16:55:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[+] Reduced XGBoost Calibration Complete. Optimal Threshold: 0.95 (Val F1: 0.9847)


In [21]:

# Ensure strict reproducibility
set_seed(42)

# Instantiate exact baseline architecture on target hardware
dnn_model_reduced = RobustNetworkSecurityDNN(input_dim=num_features_reduced).to(device)

# Prepare PyTorch Tensors on Reduced Feature Space
train_dataset_r = TensorDataset(
    torch.FloatTensor(X_train_resampled_reduced.values),
    torch.FloatTensor(y_train_resampled_reduced).unsqueeze(1)
)
val_x_tensor_r = torch.FloatTensor(X_val_reduced.values).to(device)
val_y_tensor_r = torch.FloatTensor(y_val).unsqueeze(1).to(device)

# Configure identical batching and optimization parameters
train_loader_r = DataLoader(train_dataset_r, batch_size=512, shuffle=True)
criterion = nn.BCELoss()
optimizer_r = optim.Adam(dnn_model_reduced.parameters(), lr=0.001)

patience = 10
best_val_loss_r = float('inf')
best_model_weights_r = None
patience_counter_r = 0
max_epochs = 150

print("[*] Initiating Reduced DNN optimization loop (Identical pipeline to baseline)...")
for epoch in range(1, max_epochs + 1):
    dnn_model_reduced.train()
    running_loss = 0.0
    
    for batch_x, batch_y in train_loader_r:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer_r.zero_grad()
        outputs = dnn_model_reduced(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer_r.step()
        running_loss += loss.item() * batch_x.size(0)
        
    dnn_model_reduced.eval()
    with torch.no_grad():
        val_outputs = dnn_model_reduced(val_x_tensor_r)
        val_loss = criterion(val_outputs, val_y_tensor_r).item()
        
    if val_loss < best_val_loss_r:
        best_val_loss_r = val_loss
        best_model_weights_r = copy.deepcopy(dnn_model_reduced.state_dict())
        patience_counter_r = 0
    else:
        patience_counter_r += 1
        
    if patience_counter_r >= patience:
        print(f"[*] Early stopping triggered at Epoch {epoch}.")
        break

if best_model_weights_r is not None:
    dnn_model_reduced.load_state_dict(best_model_weights_r)
print(f"[+] Reduced DNN training complete. Best Validation Loss: {best_val_loss_r:.5f}")

[*] Initiating Reduced DNN optimization loop (Identical pipeline to baseline)...
[*] Early stopping triggered at Epoch 62.
[+] Reduced DNN training complete. Best Validation Loss: 0.04310


In [22]:

def get_dnn_reduced_probabilities(df_input):
    dnn_model_reduced.eval()
    with torch.no_grad():
        tensor_input = torch.FloatTensor(df_input.values).to(device)
        probs = dnn_model_reduced(tensor_input).cpu().numpy().flatten()
    return probs

# Grid search threshold calibration matching baseline protocol
dnn_val_probs_r = get_dnn_reduced_probabilities(X_val_reduced)
thresholds_pool = np.arange(0.50, 0.96, 0.05)

best_dnn_threshold_reduced = 0.50
best_f1_dnn_r = 0.0

for t in thresholds_pool:
    preds = (dnn_val_probs_r >= t).astype(int)
    f1 = f1_score(y_val, preds, zero_division=0)
    if f1 > best_f1_dnn_r:
        best_f1_dnn_r = f1
        best_dnn_threshold_reduced = t

print(f"[+] Calibrated Reduced DNN Threshold: {best_dnn_threshold_reduced:.2f} (Val F1: {best_f1_dnn_r:.4f})")

[+] Calibrated Reduced DNN Threshold: 0.90 (Val F1: 0.8811)


In [23]:

xgb_test_probs_r = xgb_model_reduced.predict_proba(X_test_reduced)[:, 1]
xgb_test_preds_r = (xgb_test_probs_r >= best_thresh_xgb_reduced).astype(int)

# Use helper function directly to fetch test set predictions cleanly
dnn_test_probs_r = get_dnn_reduced_probabilities(X_test_reduced)
dnn_test_preds_r = (dnn_test_probs_r >= best_dnn_threshold_reduced).astype(int)

print("\n" + "="*80)
print("             REDUCING TABULAR COLLINEARITY PREDICTIVE AUDIT OVERVIEW")
print("="*80)
print("[XGBoost Reduced Feature Space layout]")
print(classification_report(y_test, xgb_test_preds_r, digits=4))
print("-"*80)
print("[PyTorch DNN Reduced Feature Space layout]")
print(classification_report(y_test, dnn_test_preds_r, digits=4))
print("="*80)


             REDUCING TABULAR COLLINEARITY PREDICTIVE AUDIT OVERVIEW
[XGBoost Reduced Feature Space layout]
              precision    recall  f1-score   support

           0     0.9997    0.9997    0.9997     25208
           1     0.9786    0.9786    0.9786       327

    accuracy                         0.9995     25535
   macro avg     0.9892    0.9892    0.9892     25535
weighted avg     0.9995    0.9995    0.9995     25535

--------------------------------------------------------------------------------
[PyTorch DNN Reduced Feature Space layout]
              precision    recall  f1-score   support

           0     0.9989    0.9976    0.9982     25208
           1     0.8306    0.9144    0.8705       327

    accuracy                         0.9965     25535
   macro avg     0.9147    0.9560    0.9343     25535
weighted avg     0.9967    0.9965    0.9966     25535



In [24]:
# Recompute model-specific cohorts based strictly on the 39-feature retrained models
print("[*] Generating model-specific cohorts for the retrained 39-feature models...")

# 1. Reduced XGBoost Cohorts
xgb_tp_r = np.where((y_test == 1) & (xgb_test_preds_r == 1))[0]
xgb_tn_r = np.where((y_test == 0) & (xgb_test_preds_r == 0))[0]
xgb_fp_r = np.where((y_test == 0) & (xgb_test_preds_r == 1))[0]
xgb_fn_r = np.where((y_test == 1) & (xgb_test_preds_r == 0))[0]

xgb_margin_r = np.abs(xgb_test_probs_r - best_thresh_xgb_reduced)
xgb_borderline_r = np.argsort(xgb_margin_r)[:100]

model_cohorts_reduced = {
    'XGBoost': {
        'True Positives': sample_indices(xgb_tp_r),
        'True Negatives': sample_indices(xgb_tn_r),
        'False Positives': sample_indices(xgb_fp_r),
        'False Negatives': sample_indices(xgb_fn_r),
        'Borderline Cases': sample_indices(xgb_borderline_r)
    }
}

# 2. Reduced DNN Cohorts
dnn_tp_r = np.where((y_test == 1) & (dnn_test_preds_r == 1))[0]
dnn_tn_r = np.where((y_test == 0) & (dnn_test_preds_r == 0))[0]
dnn_fp_r = np.where((y_test == 0) & (dnn_test_preds_r == 1))[0]
dnn_fn_r = np.where((y_test == 1) & (dnn_test_preds_r == 0))[0]

dnn_margin_r = np.abs(dnn_test_probs_r - best_dnn_threshold_reduced)
dnn_borderline_r = np.argsort(dnn_margin_r)[:100]

model_cohorts_reduced['DNN'] = {
    'True Positives': sample_indices(dnn_tp_r),
    'True Negatives': sample_indices(dnn_tn_r),
    'False Positives': sample_indices(dnn_fp_r),
    'False Negatives': sample_indices(dnn_fn_r),
    'Borderline Cases': sample_indices(dnn_borderline_r)
}

print("=================== REDUCED MODEL-SPECIFIC COHORT COUNTS (N) ===================")
for model_name, cohorts in model_cohorts_reduced.items():
    print(f"[{model_name}]")
    for c_name, c_indices in cohorts.items():
        print(f"  -> {c_name}: N = {len(c_indices)}")
print("================================================================================")

[*] Generating model-specific cohorts for the retrained 39-feature models...
=================== REDUCED MODEL-SPECIFIC COHORT COUNTS (N) ===================
[XGBoost]
  -> True Positives: N = 50
  -> True Negatives: N = 50
  -> False Positives: N = 7
  -> False Negatives: N = 7
  -> Borderline Cases: N = 50
[DNN]
  -> True Positives: N = 50
  -> True Negatives: N = 50
  -> False Positives: N = 50
  -> False Negatives: N = 28
  -> Borderline Cases: N = 50


In [25]:
#Reduced Explainers with Continuous Setup
print("[*] Instantiating reduced explainer frameworks over the non-collinear boundary...")

# 1. Initialize Reduced LIME with Primary Continuous Setup
lime_explainer_reduced = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_reduced.values,
    feature_names=reduced_features_list,
    class_names=["Benign", "Web Attack"],
    mode="classification",
    discretize_continuous=False,
    kernel_width=2.5,
    random_state=42
)

# 2. Aligned Predict Proba Wrappers for Reduced Space
def xgb_predict_proba_reduced_wrapper(x_numpy):
    df_temp = pd.DataFrame(x_numpy, columns=reduced_features_list)
    return xgb_model_reduced.predict_proba(df_temp)

def dnn_predict_proba_reduced_wrapper(x_numpy):
    df_temp = pd.DataFrame(x_numpy, columns=reduced_features_list)
    probs_class_1 = get_dnn_reduced_probabilities(df_temp)
    probs_class_0 = 1.0 - probs_class_1
    return np.column_stack((probs_class_0, probs_class_1))

# 3. Initialize Reduced SHAP Explainers
shap_background_reduced = shap.sample(X_train_reduced, 100, random_state=42)
shap_explainer_xgb_r = shap.KernelExplainer(model=xgb_predict_proba_reduced_wrapper, data=shap_background_reduced)
shap_background_reduced_dnn = shap.sample(X_train_reduced, 100, random_state=42)
shap_explainer_dnn_r = shap.KernelExplainer(model=dnn_predict_proba_reduced_wrapper, data=shap_background_reduced_dnn)

print(f"[+] Reduced explainers initialized (Continuous LIME KW=2.5). Target dimensions: {num_features_reduced}")

[*] Instantiating reduced explainer frameworks over the non-collinear boundary...
[+] Reduced explainers initialized (Continuous LIME KW=2.5). Target dimensions: 39


In [26]:
# --- Sanity Check for Reduced DNN Wrapper ---
test_sample_2d = X_test_reduced.iloc[:5].values
test_sample_1d = X_test_reduced.iloc[0].values

probs_2d = dnn_predict_proba_reduced_wrapper(test_sample_2d)
probs_1d = dnn_predict_proba_reduced_wrapper(test_sample_1d.reshape(1, -1))

print(f"[*] 2D input batch shape: {probs_2d.shape} (Expected: (5, 2))")
print(f"[*] 1D input batch shape: {probs_1d.shape} (Expected: (1, 2))")
print(f"[*] Row sum check (all should equal 1.0): {np.allclose(probs_2d.sum(axis=1), 1.0)}")
print(f"[*] Sample Class-1 probabilities:\n    {probs_2d[:, 1].round(4)}")
assert probs_2d.shape == (5, 2), "Shape mismatch on batch inference!"
assert np.allclose(probs_2d.sum(axis=1), 1.0), "Probabilities do not sum to 1.0!"
print("[+] Wrapper sanity check passed successfully!")

[*] 2D input batch shape: (5, 2) (Expected: (5, 2))
[*] 1D input batch shape: (1, 2) (Expected: (1, 2))
[*] Row sum check (all should equal 1.0): True
[*] Sample Class-1 probabilities:
    [0.     0.     0.     0.     0.9923]
[+] Wrapper sanity check passed successfully!


In [27]:
#Reduced Space Attribution Extraction
reduced_agreement_records = []
print("[*] Extracting evaluations across pruned model structures...")

reduced_configs = [
    ('XGBoost', xgb_predict_proba_reduced_wrapper, shap_explainer_xgb_r),
    ('DNN', dnn_predict_proba_reduced_wrapper, shap_explainer_dnn_r)
]

for model_label, predict_fn, shap_exp in reduced_configs:
    print(f"\n[*] Evaluating reduced-space cohorts for: {model_label}")
    for cohort_name, indices in model_cohorts_reduced[model_label].items():
        if not indices:
            continue
        sample_count = len(indices)
        print(f"    -> Auditing cohort: {cohort_name} (N = {sample_count})...")
        
        for test_idx in indices:
            instance_vector_r = X_test_reduced.iloc[test_idx].values
            
            exp_l = lime_explainer_reduced.explain_instance(
                data_row=instance_vector_r, 
                predict_fn=predict_fn, 
                labels=(1,),
                num_features=num_features_reduced
            )
            lime_vec = np.zeros(num_features_reduced)
            for f_idx, w in exp_l.local_exp[1]: 
                lime_vec[f_idx] = w
            
            shap_raw = shap_exp.shap_values(instance_vector_r.reshape(1, -1), nsamples=100)
            if isinstance(shap_raw, list):
                shap_vec = shap_raw[1].flatten()
            elif isinstance(shap_raw, np.ndarray) and len(shap_raw.shape) == 3:
                shap_vec = shap_raw[0, :, 1]
            else:
                shap_vec = shap_raw.flatten()
                
            assert len(lime_vec) == num_features_reduced, f"LIME vector size mismatch: {len(lime_vec)}"
            assert len(shap_vec) == num_features_reduced, f"SHAP vector size mismatch: {len(shap_vec)}"
            
            row_metrics = {
                'Cohort': cohort_name,
                'N_Samples': sample_count,
                'Model': model_label,
                'Test_Index': test_idx,
                'LIME_R2': exp_l.score
            }
            
            for k in [3, 5, 10]:
                top_lime = set(np.argsort(np.abs(lime_vec))[-k:])
                top_shap = set(np.argsort(np.abs(shap_vec))[-k:])
                
                overlap = len(top_lime.intersection(top_shap))
                union_set = top_lime.union(top_shap)
                jaccard = overlap / len(union_set) if len(union_set) > 0 else 0.0
                
                row_metrics[f'Overlap_{k}'] = overlap
                row_metrics[f'Jaccard_{k}'] = jaccard
                
                union_indices = list(union_set)
                slice_l = lime_vec[union_indices]
                slice_s = shap_vec[union_indices]
                
                if np.std(slice_l) == 0 or np.std(slice_s) == 0 or len(union_indices) < 2:
                    row_metrics[f'Union_Spearman_{k}'] = 0.0
                else:
                    rho, _ = stats.spearmanr(slice_l, slice_s)
                    row_metrics[f'Union_Spearman_{k}'] = 0.0 if np.isnan(rho) else rho
                
            reduced_agreement_records.append(row_metrics)

df_reduced_agreement = pd.DataFrame(reduced_agreement_records)
print("[+] Reduced space evaluation complete.")

[*] Extracting evaluations across pruned model structures...

[*] Evaluating reduced-space cohorts for: XGBoost
    -> Auditing cohort: True Positives (N = 50)...


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

[+] Reduced space evaluation complete.


In [28]:
#Master Reduced Space Report
reduced_report_rows = []

for (cohort_name, model_type), group_df in df_reduced_agreement.groupby(['Cohort', 'Model']):
    sample_size = group_df['N_Samples'].iloc[0]
    row_summary = {
        'Cohort': cohort_name, 
        'N_Samples': sample_size,
        'Model': model_type
    }
    for metric in ['LIME_R2', 'Overlap_3', 'Overlap_5', 'Overlap_10', 'Jaccard_5', 'Jaccard_10', 'Union_Spearman_10']:
        mean_v, low_v, high_v = compute_bootstrapped_ci(group_df[metric], n_bootstrap=5000)
        row_summary[metric] = f"{mean_v:.3f} [{low_v:.3f}, {high_v:.3f}]"
    reduced_report_rows.append(row_summary)

df_reduced_master_report = pd.DataFrame(reduced_report_rows)

print("\n" + "="*160)
print("             FINAL MASTER REPORT: EXPLAINER AGREEMENT OVER PRUNED NON-COLLINEAR MODELS (WITH 95% CIs)")
print("="*160)
display(df_reduced_master_report)
print("="*160)


             FINAL MASTER REPORT: EXPLAINER AGREEMENT OVER PRUNED NON-COLLINEAR MODELS (WITH 95% CIs)


,Cohort,N_Samples,Model,LIME_R2,Overlap_3,Overlap_5,Overlap_10,Jaccard_5,Jaccard_10,Union_Spearman_10
0,Borderline Cases,50,DNN,"0.438 [0.433, 0.444]","0.240 [0.120, 0.360]","0.860 [0.660, 1.060]","4.000 [3.660, 4.320]","0.101 [0.076, 0.128]","0.257 [0.231, 0.284]","0.465 [0.404, 0.528]"
1,Borderline Cases,50,XGBoost,"0.130 [0.107, 0.168]","0.840 [0.700, 0.980]","1.820 [1.680, 1.960]","4.780 [4.400, 5.160]","0.227 [0.206, 0.249]","0.324 [0.292, 0.357]","0.193 [0.093, 0.293]"
2,False Negatives,28,DNN,"0.437 [0.433, 0.441]","0.179 [0.036, 0.321]","0.821 [0.643, 1.000]","3.714 [3.214, 4.250]","0.092 [0.071, 0.113]","0.238 [0.199, 0.284]","0.407 [0.308, 0.497]"
3,False Negatives,7,XGBoost,"0.126 [0.080, 0.189]","0.714 [0.429, 1.000]","1.571 [1.143, 2.143]","4.714 [3.857, 5.429]","0.196 [0.131, 0.287]","0.314 [0.245, 0.376]","-0.087 [-0.268, 0.099]"
4,False Positives,50,DNN,"0.439 [0.436, 0.443]","0.160 [0.060, 0.260]","0.960 [0.760, 1.160]","4.620 [4.320, 4.940]","0.114 [0.089, 0.140]","0.308 [0.281, 0.335]","0.494 [0.435, 0.553]"
5,False Positives,7,XGBoost,"0.092 [0.069, 0.116]","0.286 [0.000, 0.571]","1.571 [0.857, 2.143]","5.571 [4.857, 6.143]","0.200 [0.103, 0.301]","0.392 [0.325, 0.452]","0.340 [0.199, 0.474]"
6,True Negatives,50,DNN,"0.414 [0.393, 0.428]","0.300 [0.180, 0.440]","0.880 [0.680, 1.100]","3.200 [2.860, 3.540]","0.104 [0.078, 0.132]","0.197 [0.173, 0.222]","-0.243 [-0.322, -0.162]"
7,True Negatives,50,XGBoost,"0.084 [0.078, 0.088]","1.380 [1.200, 1.560]","3.000 [2.740, 3.260]","5.420 [5.160, 5.680]","0.457 [0.401, 0.520]","0.378 [0.352, 0.403]","0.009 [-0.057, 0.071]"
8,True Positives,50,DNN,"0.441 [0.434, 0.449]","0.240 [0.140, 0.360]","0.940 [0.720, 1.160]","4.060 [3.720, 4.400]","0.112 [0.085, 0.139]","0.262 [0.234, 0.291]","0.458 [0.387, 0.524]"
9,True Positives,50,XGBoost,"0.144 [0.115, 0.187]","0.980 [0.880, 1.060]","1.620 [1.480, 1.760]","3.940 [3.640, 4.240]","0.198 [0.176, 0.219]","0.251 [0.228, 0.276]","0.400 [0.324, 0.469]"
